# Stage 4: Transaction-Level Data Preparation
This notebook fulfills the practical deliverable for Stage 4: combining our relational tables into a single flat dataset (one row per transaction) with historical features and a clearly defined outcome label, strictly preventing data leakage.

## 1. Connecting to Database

In [10]:
import pandas as pd
import sqlite3

# Connect to the destination database
conn = sqlite3.connect('destination_db.sqlite')


## 2. Feature Engineering & Leakage-Free SQL
We will construct a single SQL query that:
1. Calculates `days_since_signup`.
2. Uses a Window Function to calculate `previous_transaction_count` (strictly using `ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING` to avoid leakage).
3. Assigns `is_fraud` = 1 only if the chargeback reason indicates fraud.

In [16]:
query = '''
SELECT 
    transaction_id,
    transaction_amount,
    customer_risk_score,
    days_since_signup,
    COALESCE(
        COUNT(transaction_id) OVER(
            PARTITION BY customer_id 
            ORDER BY date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), 0
    ) as previous_transaction_count,
    is_fraud
FROM (
    SELECT 
        t.id as transaction_id,
        t.date,
        c.id as customer_id,
        t.amount as transaction_amount,
        c.risk_score as customer_risk_score,
        CAST(julianday(t.date) - julianday(c.signup_date) AS INTEGER) as days_since_signup,
        COALESCE(MAX(CASE WHEN cb.reason IN ('Fraudulent transaction', 'Card Stolen') THEN 1 ELSE 0 END), 0) as is_fraud
    FROM imported_transactions t
    JOIN imported_customers c ON t.customer_id = c.id
    LEFT JOIN imported_chargebacks cb ON t.id = cb.transaction_id
    GROUP BY t.id
)
ORDER BY date ASC
'''

# Execute the query and load into a pandas DataFrame
df_training = pd.read_sql_query(query, conn)
#print(df_training.head(10)) - without formating
display(df_training.head(10))
#print(df_training.shape)
#print(df_training.info)


,transaction_id,transaction_amount,customer_risk_score,days_since_signup,previous_transaction_count,is_fraud
0,489,188.94,0.04,10,0,0
1,47,45.35,0.25,3,0,0
2,110,58.05,0.04,4,0,0
3,430,258.21,0.1,15,0,0
4,324,37.11,0.25,30,1,0
5,86,84.25,0.24,10,0,0
6,336,69.35,0.25,55,2,0
7,335,180.91,0.09,70,0,0
8,280,365.05,0.02,82,0,0
9,28,49.92,0.01,1,0,0


## 3. Verification & Validation

In [18]:
# Verify exactly one row per transaction
total_rows = df_training.shape[0]
unique_transactions = df_training['transaction_id'].nunique()
#total_rows=500
print(f"Total Rows: {total_rows}")
print(f"Unique Transactions: {unique_transactions}")
assert total_rows == unique_transactions, "Error: Multiple rows exist for a single transaction!"
print("✅ Passed: Exactly one row per transaction.")

# Check the label distribution
print("\nLabel Distribution (is_fraud):")
print(df_training['is_fraud'].value_counts())


Total Rows: 500
Unique Transactions: 500
✅ Passed: Exactly one row per transaction.

Label Distribution (is_fraud):
is_fraud
0    480
1     20
Name: count, dtype: int64


## 4. Exporting to CSV
Now that our dataset is verified, we will export it as `training_dataset.csv` for Stage 5 (Modeling).

In [19]:
# Export to CSV without the index column
df_training.to_csv('training_dataset.csv', index=False)
print("Successfully exported to training_dataset.csv!")

# Close the database connection
conn.close()


Successfully exported to training_dataset.csv!
